<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎨 Krea-2-Turbo - Fast Text-to-Image Generator</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition - Created by <strong>AIQUEST Academy</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>High-Speed 12B DiT Generation | 8-Step Distilled INT8 Inference</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Kaggle-T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
</div>

---

### What is this notebook?

**Fast Text-to-Image Generation** using the **Krea-2-Turbo** model (12B parameters) powered by the **Wan2GP** engine and **mmgp** offloading.

- **GPU Configuration:** GPU T4 x2 (dual T4 GPUs, 2x16GB VRAM)
- **Performance Optimization**: Natively runs in **INT8** quantization on Turing Tensor Cores, bypassing the PCIe cross-communication bottlenecks of Diffusers.
- **Inference Steps**: 8 steps (distilled turbo architecture)

### Quick Start
1. **Settings -> Accelerator -> GPU T4 x2** (Dual T4)
2. **Turn on Internet** in the Settings sidebar
3. Run all cells in order
4. Use the public Gradio link from Cell 4 to open the web interface


---
## Step 1: Environment Setup

Optimizes memory for Kaggle GPU T4 x2 (~30GB RAM, 32GB VRAM combined).

In [ ]:
import os, gc, psutil

print('=== Kaggle GPU T4 x2 Environment Setup ===')
print(f'RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available')

os.system('echo 3 | sudo tee /proc/sys/vm/drop_caches > /dev/null 2>&1')
os.system('echo 1 | sudo tee /proc/sys/vm/overcommit_memory > /dev/null 2>&1')
gc.collect()

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,garbage_collection_threshold:0.6'
os.environ['MALLOC_TRIM_THRESHOLD_'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print('✅ Environment optimized!')

---
## Step 2: Clone Wan2GP & Install Dependencies

In [ ]:
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
    print('GPU Active!')
except Exception:
    print('WARNING: No GPU. Go to Settings -> Accelerator -> GPU T4 x2')

# Clone the Wan2GP codebase
!git clone https://github.com/DeepBeepMeep/Wan2GP.git

# Install Wan2GP requirements and mmgp/gradio dependencies
!pip install --timeout 120 --retries 5 -q -r Wan2GP/requirements.txt
!pip install --timeout 120 --retries 5 -q mmgp gradio

# Patch Wan2GP's hardcoded dtype = torch.bfloat16 and cast weights to float16 to prevent dtype mismatches on T4 GPUs
import os
krea_main_path = "Wan2GP/models/krea2/krea2_main.py"
if os.path.exists(krea_main_path):
    with open(krea_main_path, "r") as f:
        content = f.read()
    content = content.replace('\r\n', '\n')
    
    # 1. Comment out dtype = torch.bfloat16 override
    if "dtype = torch.bfloat16" in content:
        content = content.replace("dtype = torch.bfloat16", "# dtype = torch.bfloat16 # Patched for FP16 speedup on T4")
        
    # 2. Add tf_preprocess to cast loaded floating point weights to dtype
    old_tf = "offload.load_model_data(transformer, model_filename, writable_tensors=False, preprocess_sd=preprocess_sd, default_dtype=dtype)"
    new_tf = """def tf_preprocess(sd):
        sd = preprocess_sd(sd)
        return {k: v.to(dtype) if (isinstance(v, torch.Tensor) and v.is_floating_point()) else v for k, v in sd.items()}
    offload.load_model_data(transformer, model_filename, writable_tensors=False, preprocess_sd=tf_preprocess, default_dtype=dtype)"""
    if old_tf in content:
        content = content.replace(old_tf, new_tf)
        
    # 3. Add te_preprocess
    old_te = "offload.load_model_data(text_encoder.language_model, text_encoder_filename, modelPrefix=\"language_model\", writable_tensors=False, default_dtype=dtype)"
    new_te = """def te_preprocess(sd):
        return {k: v.to(dtype) if (isinstance(v, torch.Tensor) and v.is_floating_point()) else v for k, v in sd.items()}
    offload.load_model_data(text_encoder.language_model, text_encoder_filename, modelPrefix="language_model", writable_tensors=False, preprocess_sd=te_preprocess, default_dtype=dtype)"""
    if old_te in content:
        content = content.replace(old_te, new_te)
        
    # 4. Add vae_preprocess
    old_vae = "offload.load_model_data(vae, filename, writable_tensors=False, default_dtype=dtype)"
    new_vae = """def vae_preprocess(sd):
        return {k: v.to(dtype) if (isinstance(v, torch.Tensor) and v.is_floating_point()) else v for k, v in sd.items()}
    offload.load_model_data(vae, filename, writable_tensors=False, preprocess_sd=vae_preprocess, default_dtype=dtype)"""
    if old_vae in content:
        content = content.replace(old_vae, new_vae)
        
    # 5. Add Conditioning Rebalance patch to Krea2Pipeline.__call__ and generate
    if "def __call__(self, prompts, negative_prompts=None, width=1024, height=1024, steps=28, guidance=4.5, seed=0, y1=0.5, y2=1.15, mu=None, callback=None, loras_slists=None):" in content:
        content = content.replace(
            "def __call__(self, prompts, negative_prompts=None, width=1024, height=1024, steps=28, guidance=4.5, seed=0, y1=0.5, y2=1.15, mu=None, callback=None, loras_slists=None):",
            "def __call__(self, prompts, negative_prompts=None, width=1024, height=1024, steps=28, guidance=4.5, seed=0, y1=0.5, y2=1.15, mu=None, callback=None, loras_slists=None, rebalance_weights=None):"
        )
        
    old_call_prepare = """        txt, txtmask = self._encode_prompts(prompts, device, dtype)
        if txt is None:
            return None
        x, pos, mask = _prepare(noise, txt.shape[1], patch, txtmask)
        cfg = guidance > 0
        if cfg:
            untxt, untxtmask = self._encode_prompts(negative_prompts, device, dtype)
            if untxt is None:
                return None"""
                
    new_call_prepare = """        txt, txtmask = self._encode_prompts(prompts, device, dtype)
        if txt is None:
            return None
        if rebalance_weights is not None:
            print(f"-> [Wan2GP Pipeline] Applying conditioning layer weights rebalance...")
            txt = txt * rebalance_weights.to(device=device, dtype=dtype)
        x, pos, mask = _prepare(noise, txt.shape[1], patch, txtmask)
        cfg = guidance > 0
        if cfg:
            untxt, untxtmask = self._encode_prompts(negative_prompts, device, dtype)
            if untxt is None:
                return None
            if rebalance_weights is not None:
                untxt = untxt * rebalance_weights.to(device=device, dtype=dtype)"""
                
    if old_call_prepare in content:
        content = content.replace(old_call_prepare, new_call_prepare)
        
    if "def generate(" in content and "rebalance_weights=None" not in content:
        content = content.replace(
            "        loras_slists=None,\n        **kwargs,",
            "        loras_slists=None,\n        rebalance_weights=None,\n        **kwargs,"
        )
        content = content.replace(
            "callback=callback, loras_slists=loras_slists)",
            "callback=callback, loras_slists=loras_slists, rebalance_weights=rebalance_weights)"
        )
        
    with open(krea_main_path, "w") as f:
        f.write(content)
    print("✅ Patched Wan2GP/models/krea2/krea2_main.py successfully for speed and dtype compatibility!")
else:
    print("⚠️ Wan2GP/models/krea2/krea2_main.py not found. If this is a dry run before cloning, that's fine.")

print('✅ Dependencies installed and patched!')

---
## Step 3: Download INT8 Models

Downloads the INT8 quantized Krea-2-Turbo model weights, text encoder, and VAE configuration.

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

REPO = "DeepBeepMeep/krea-2"
QWEN_IMAGE_REPO = "DeepBeepMeep/Qwen_image"
MODEL_DIR = "Wan2GP/models"
TMP_DIR = "/kaggle/tmp/models"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

# 1. Download Transformer (quanto int8 - ~12.5 GB) directly to internal storage
TRANSFORMER_FILE = "Krea2Turbo_quanto_bf16_int8.safetensors"
dest_transformer = os.path.join(MODEL_DIR, TRANSFORMER_FILE)
if not os.path.exists(dest_transformer):
    print(f"Downloading {TRANSFORMER_FILE} to {MODEL_DIR} (internal storage)...")
    hf_hub_download(repo_id=REPO, filename=TRANSFORMER_FILE, local_dir=MODEL_DIR, local_dir_use_symlinks=False)
    print(f"  ✓ {TRANSFORMER_FILE} (downloaded directly)")
else:
    print(f"  ✓ Already exists: {TRANSFORMER_FILE}")

# 2. Download Text Encoder (Qwen3-VL-4B-Instruct - ~4 GB) to /kaggle/tmp and symlink
TEXT_ENCODER_FOLDER = "Qwen3-VL-4B-Instruct"
TEXT_ENCODER_FILES = [
    "Qwen3-VL-4B-Instruct_quanto_bf16_int8.safetensors",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "chat_template.jinja"
]
encoder_dest_dir = os.path.join(MODEL_DIR, TEXT_ENCODER_FOLDER)
encoder_tmp_dir = os.path.join(TMP_DIR, TEXT_ENCODER_FOLDER)
os.makedirs(encoder_tmp_dir, exist_ok=True)

if not os.path.exists(encoder_dest_dir):
    for f in TEXT_ENCODER_FILES:
        dest_file = os.path.join(encoder_tmp_dir, f)
        if not os.path.exists(dest_file):
            print(f"Downloading text_encoder/{f} → /kaggle/tmp ...")
            hf_hub_download(repo_id=REPO, filename=f"{TEXT_ENCODER_FOLDER}/{f}", local_dir=TMP_DIR, local_dir_use_symlinks=False)
    os.symlink(encoder_tmp_dir, encoder_dest_dir)
    print(f"  ✓ {TEXT_ENCODER_FOLDER}/ (symlinked)")
else:
    print(f"  ✓ Already exists: {TEXT_ENCODER_FOLDER}/")

# 3. Download VAE (Qwen Image VAE - ~1.5 GB) to /kaggle/tmp and symlink
VAE_FILES = ["qwen_vae.safetensors", "qwen_vae_config.json"]
for f in VAE_FILES:
    dest_file = os.path.join(MODEL_DIR, f)
    if not os.path.exists(dest_file):
        tmp_file = os.path.join(TMP_DIR, f)
        if not os.path.exists(tmp_file):
            print(f"Downloading VAE/{f} → /kaggle/tmp ...")
            hf_hub_download(repo_id=QWEN_IMAGE_REPO, filename=f, local_dir=TMP_DIR, local_dir_use_symlinks=False)
        os.symlink(tmp_file, dest_file)
        print(f"  ✓ {f} (symlinked)")
    else:
        print(f"  ✓ Already exists: {f}")

# Cleanup HF cache to save space
for d in [os.path.join(MODEL_DIR, '.cache'), os.path.join(TMP_DIR, '.cache'), '/kaggle/tmp/hf_home']:
    if os.path.exists(d):
        try:
            shutil.rmtree(d)
        except Exception:
            pass

os.system('df -h /kaggle/working /kaggle/tmp')
print('\n✅ All downloads complete!')

---
## Step 4: Write Generation Script & Launch UI

In [ ]:
%%writefile run_krea_turbo.py
import gc
import os
import sys
import time
import random
import traceback
import numpy as np
from PIL import Image

# ---- bootstrap Wan2GP ----
WAN2GP_DIR = os.path.abspath("Wan2GP")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

# Patch the hardcoded dtype and loader preprocessing in krea2_main.py before importing handler to ensure FP16 Tensor Core acceleration on T4
krea_main_rel_path = os.path.join("models", "krea2", "krea2_main.py")
if os.path.exists(krea_main_rel_path):
    with open(krea_main_rel_path, "r") as f:
        content = f.read()
    content = content.replace('\r\n', '\n')
    
    if "dtype = torch.bfloat16" in content:
        print("Patching krea2_main.py: commenting out hardcoded bfloat16 override...")
        content = content.replace("dtype = torch.bfloat16", "# dtype = torch.bfloat16 # Patched for FP16 speedup on T4")
        
    old_tf = "offload.load_model_data(transformer, model_filename, writable_tensors=False, preprocess_sd=preprocess_sd, default_dtype=dtype)"
    new_tf = """def tf_preprocess(sd):
        sd = preprocess_sd(sd)
        return {k: v.to(dtype) if (isinstance(v, torch.Tensor) and v.is_floating_point()) else v for k, v in sd.items()}
    offload.load_model_data(transformer, model_filename, writable_tensors=False, preprocess_sd=tf_preprocess, default_dtype=dtype)"""
    if old_tf in content:
        print("Patching transformer loading preprocessing in krea2_main.py...")
        content = content.replace(old_tf, new_tf)
        
    old_te = "offload.load_model_data(text_encoder.language_model, text_encoder_filename, modelPrefix=\"language_model\", writable_tensors=False, default_dtype=dtype)"
    new_te = """def te_preprocess(sd):
        return {k: v.to(dtype) if (isinstance(v, torch.Tensor) and v.is_floating_point()) else v for k, v in sd.items()}
    offload.load_model_data(text_encoder.language_model, text_encoder_filename, modelPrefix="language_model", writable_tensors=False, preprocess_sd=te_preprocess, default_dtype=dtype)"""
    if old_te in content:
        print("Patching text encoder loading preprocessing in krea2_main.py...")
        content = content.replace(old_te, new_te)
        
    old_vae = "offload.load_model_data(vae, filename, writable_tensors=False, default_dtype=dtype)"
    new_vae = """def vae_preprocess(sd):
        return {k: v.to(dtype) if (isinstance(v, torch.Tensor) and v.is_floating_point()) else v for k, v in sd.items()}
    offload.load_model_data(vae, filename, writable_tensors=False, preprocess_sd=vae_preprocess, default_dtype=dtype)"""
    if old_vae in content:
        print("Patching VAE loading preprocessing in krea2_main.py...")
        content = content.replace(old_vae, new_vae)
        
    # 5. Add Conditioning Rebalance patch to Krea2Pipeline.__call__ and generate
    if "def __call__(self, prompts, negative_prompts=None, width=1024, height=1024, steps=28, guidance=4.5, seed=0, y1=0.5, y2=1.15, mu=None, callback=None, loras_slists=None):" in content:
        print("Patching Krea2Pipeline.__call__ to support rebalance weights...")
        content = content.replace(
            "def __call__(self, prompts, negative_prompts=None, width=1024, height=1024, steps=28, guidance=4.5, seed=0, y1=0.5, y2=1.15, mu=None, callback=None, loras_slists=None):",
            "def __call__(self, prompts, negative_prompts=None, width=1024, height=1024, steps=28, guidance=4.5, seed=0, y1=0.5, y2=1.15, mu=None, callback=None, loras_slists=None, rebalance_weights=None):"
        )
        
    old_call_prepare = """        txt, txtmask = self._encode_prompts(prompts, device, dtype)
        if txt is None:
            return None
        x, pos, mask = _prepare(noise, txt.shape[1], patch, txtmask)
        cfg = guidance > 0
        if cfg:
            untxt, untxtmask = self._encode_prompts(negative_prompts, device, dtype)
            if untxt is None:
                return None"""
                
    new_call_prepare = """        txt, txtmask = self._encode_prompts(prompts, device, dtype)
        if txt is None:
            return None
        if rebalance_weights is not None:
            print(f"-> [Wan2GP Pipeline] Applying conditioning layer weights rebalance...")
            txt = txt * rebalance_weights.to(device=device, dtype=dtype)
        x, pos, mask = _prepare(noise, txt.shape[1], patch, txtmask)
        cfg = guidance > 0
        if cfg:
            untxt, untxtmask = self._encode_prompts(negative_prompts, device, dtype)
            if untxt is None:
                return None
            if rebalance_weights is not None:
                untxt = untxt * rebalance_weights.to(device=device, dtype=dtype)"""
                
    if old_call_prepare in content:
        content = content.replace(old_call_prepare, new_call_prepare)
        
    if "def generate(" in content and "rebalance_weights=None" not in content:
        print("Patching Krea2.generate to accept and pass rebalance weights...")
        content = content.replace(
            "        loras_slists=None,\n        **kwargs,",
            "        loras_slists=None,\n        rebalance_weights=None,\n        **kwargs,"
        )
        content = content.replace(
            "callback=callback, loras_slists=loras_slists)",
            "callback=callback, loras_slists=loras_slists, rebalance_weights=rebalance_weights)"
        )
        
    with open(krea_main_rel_path, "w") as f:
        f.write(content)
    print("✅ Patched krea2_main.py successfully for speed and dtype compatibility!")
else:
    print("⚠️ Warning: krea2_main.py not found at expected path.")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import gradio as gr
from mmgp import offload
from shared.utils import files_locator as fl
from models.krea2.krea2_handler import family_handler

fl.set_checkpoints_paths(["models", "ckpts", "."])

# ==== LOAD INT8 QUANTIZED MODEL VIA WAN2GP ====
print("\nLoading Krea-2-Turbo model (quanto int8)...")
sys.stdout.flush()

base_model_type = "krea2_turbo"
model_def = family_handler.query_model_def(base_model_type, {})

transformer_path = os.path.join("models", "Krea2Turbo_quanto_bf16_int8.safetensors")
text_encoder_path = os.path.join("models", "Qwen3-VL-4B-Instruct", "Qwen3-VL-4B-Instruct_quanto_bf16_int8.safetensors")

if not os.path.exists(transformer_path):
    raise FileNotFoundError(f"Transformer not found at {transformer_path}")
if not os.path.exists(text_encoder_path):
    raise FileNotFoundError(f"Text encoder not found at {text_encoder_path}")

krea_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type="krea2_turbo",
    base_model_type=base_model_type,
    model_def=model_def,
    quantizeTransformer=False,
    dtype=torch.float16,
    VAE_dtype=torch.float16,
    text_encoder_filename=text_encoder_path,
)

# ==== Apply mmgp profile ====
print("\nApplying mmgp custom profile (pinned + async + HIGH budget)...")
sys.stdout.flush()

offload.profile(
    pipe,
    profile_no=2,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    pinnedMemory=True,
    asyncTransfers=True,
    budgets={
        "transformer": 13000,
        "text_encoder": 4500,
        "vae": 2000,
        "*": 1000,
    }
)
offload.shared_state["_attention"] = "sdpa"
print("✅ mmgp offloading initialized!")
sys.stdout.flush()

# ==== PROMPT ENHANCER LOGIC ====
def expand_and_enhance_prompt(prompt):
    if not prompt:
        return ""
    prompt_lower = prompt.lower()
    
    # Avoid double enhancement if already highly detailed
    if any(k in prompt_lower for k in ["photorealistic", "hyperrealistic", "highly detailed", "8k resolution"]):
        return prompt
        
    # Categories
    is_portrait = any(w in prompt_lower for w in ["person", "woman", "man", "girl", "boy", "portrait", "face", "model", "lady", "guy"])
    is_sci_fi = any(w in prompt_lower for w in ["cyberpunk", "futuristic", "sci-fi", "robot", "spaceship", "alien", "neon", "technology"])
    is_landscape = any(w in prompt_lower for w in ["forest", "mountain", "ocean", "nature", "landscape", "lake", "river", "sky", "view", "sunset", "sunrise"])
    is_fantasy = any(w in prompt_lower for w in ["dragon", "magic", "wizard", "elf", "fairy", "castle", "mythical", "ancient", "fantasy"])
    is_anime_art = any(w in prompt_lower for w in ["anime", "illustration", "drawing", "painting", "art", "sketch", "digital art", "vector"])

    import random
    
    if is_portrait:
        modifiers = [
            "captured on 85mm lens, f/1.8, natural skin texture, realistic catchlights in eyes, professional studio lighting, depth of field, stunning portrait, highly detailed, 8k",
            "candid photography, warm natural light, soft shadows, detailed facial features, cinematic color grading, photorealistic, masterpiece",
            "dramatic side-lighting, highly detailed skin pores, close-up shot, portra 400 film style, high resolution, sharp focus, exquisite composition"
        ]
        suffix = random.choice(modifiers)
    elif is_sci_fi:
        modifiers = [
            "cyberpunk aesthetic, neon glow, wet pavement with reflections, volumetric fog, high-tech details, dark moody atmosphere, cinematic lighting, ray tracing, 8k resolution",
            "futuristic design, industrial sci-fi details, cinematic composition, metallic surfaces, sharp contrast, volumetric light rays, octane render style, highly detailed",
            "dystopian atmosphere, high-tech elements, dramatic shadows, glowing circuits, ultra-detailed, cinematic wide shot, photorealistic, masterfully crafted"
        ]
        suffix = random.choice(modifiers)
    elif is_landscape:
        modifiers = [
            "golden hour sunset lighting, national geographic style, epic scale, volumetric mist, wide-angle lens, breathtaking scenery, sharp focus, highly detailed, 8k",
            "dramatic skies, natural lighting, crisp details, long exposure water reflections, gorgeous composition, masterpiece, photorealistic, ultra-detailed landscape",
            "atmospheric haze, majestic mountain peaks, rays of sun piercing through clouds, highly detailed textures, vibrant colors, stunning nature photography"
        ]
        suffix = random.choice(modifiers)
    elif is_fantasy:
        modifiers = [
            "mythical atmosphere, glowing magical particles, ethereal light, highly detailed fantasy illustration, masterpiece, epic scale, whimsical, digital painting, 8k",
            "dark fantasy style, dramatic moody lighting, ancient runic details, volumetric rays, gorgeous composition, hyper-detailed, stunning fantasy art",
            "enchanted forest lighting, magical glow, intricate details, breathtaking fantasy landscape, majestic, high-resolution digital masterpiece"
        ]
        suffix = random.choice(modifiers)
    elif is_anime_art:
        modifiers = [
            "stunning digital painting, vibrant color palette, highly detailed illustration, clean line art, beautiful lighting, anime key visual style, masterpiece",
            "concept art illustration, detailed background, soft lighting, dramatic composition, masterfully painted, artist station style, gorgeous visual",
            "vector illustration, clean details, smooth gradients, modern digital art style, highly detailed, creative graphic design"
        ]
        suffix = random.choice(modifiers)
    else:
        modifiers = [
            "highly detailed, photorealistic, cinematic lighting, masterpiece, stunning visual, sharp focus, 8k resolution, intricate textures",
            "professional photography, depth of field, dramatic lighting, rich colors, soft shadows, sharp focus, highly detailed, 8k",
            "award-winning photography, volumetric lighting, hyper-realistic, photorealistic, exquisite details, gorgeous composition, warm glow"
        ]
        suffix = random.choice(modifiers)
        
    return f"{prompt}, {suffix}"

# ==== GRADIO ACTION HELPER ====
def enhance_prompt_only_if_checked(prompt, enhance_prompt, style_preset):
    active_prompt = prompt if prompt else ""
    if enhance_prompt:
        active_prompt = expand_and_enhance_prompt(active_prompt)
    if style_preset and style_preset != "None":
        style_modifiers = {
            "Cinematic": "cinematic lighting, dramatic shadows, 8k resolution, highly detailed, film grain, masterpiece",
            "Photographic": "professional photography, 35mm lens, f/2.8, depth of field, natural lighting, highly detailed, photorealistic",
            "Anime": "anime key visual, vibrant color palette, clean line art, highly detailed digital illustration, masterpiece",
            "Cyberpunk": "cyberpunk aesthetic, glowing neon lights, futuristic city streets, volumetric smoke, high contrast, ray tracing",
            "Fantasy": "mythical fantasy landscape, glowing magical particles, ethereal light, whimsical details, digital painting, masterpiece"
        }
        modifier = style_modifiers[style_preset]
        if modifier not in active_prompt:
            if active_prompt:
                active_prompt = f"{active_prompt}, {modifier}"
            else:
                active_prompt = modifier
    return active_prompt

# ==== GENERATION FUNCTION ====
@torch.inference_mode()
def generate_image(prompt, negative_prompt, steps, aspect_ratio, resolution, seed, num_images, bypass_safety, bypass_multiplier, rebalance_weights, progress=gr.Progress(track_tqdm=True)):
    try:
        gc.collect(); torch.cuda.empty_cache()
        
        # Yield clear gallery/status instantly with selected_index=None
        yield gr.update(value=None, selected_index=None), "⏳ Initializing generation...", gr.update(visible=False)
        
        # Resolve base size
        base_size = 1024
        if resolution == "1536px (High)":
            base_size = 1536
        elif resolution == "2048px (2K Ultra)":
            base_size = 2048
            
        # Resolve dimensions based on aspect ratio (must be divisible by 16)
        if aspect_ratio == "1:1 Square":
            width, height = base_size, base_size
        elif aspect_ratio == "16:9 Landscape":
            width = base_size
            height = int((base_size * 9 / 16) // 16 * 16)
        elif aspect_ratio == "9:16 Portrait":
            width = int((base_size * 9 / 16) // 16 * 16)
            height = base_size
        elif aspect_ratio == "4:3 Standard":
            width = base_size
            height = int((base_size * 3 / 4) // 16 * 16)
        elif aspect_ratio == "3:4 Portrait":
            width = int((base_size * 3 / 4) // 16 * 16)
            height = base_size
            
        # Resolve seed
        if seed is None or seed < 0:
            initial_seed = int(torch.randint(0, 2**32 - 1, (1,)).item())
        else:
            initial_seed = int(seed)
            
        # Parse rebalance weights
        weights_tensor = None
        if bypass_safety:
            try:
                # Default weights to ones (uniform) if empty
                w_list = [1.0] * 12
                if rebalance_weights.strip():
                    w_list = [float(w.strip()) for w in rebalance_weights.split(",") if w.strip()]
                    if len(w_list) < 12:
                        w_list = w_list + [1.0] * (12 - len(w_list))
                    elif len(w_list) > 12:
                        w_list = w_list[:12]
                
                # Apply global multiplier scaling directly to layer gains
                w_list = [w * float(bypass_multiplier) for w in w_list]
                weights_tensor = torch.tensor(w_list, dtype=torch.float16).view(1, 1, 12, 1)
                print(f"⚖️ Safety filter bypass enabled! Applied conditioning layer weights scaled by {bypass_multiplier}: {w_list}")
            except Exception as w_e:
                print(f"❌ Error parsing rebalance weights: {w_e}")
        else:
            print("🔓 Safety filter bypass is disabled. Model conditioning running natively.")
                
        images = []
        seeds_used = []
        start_time = time.time()
        
        num_images = int(num_images)
        for idx in range(num_images):
            current_seed = (initial_seed + idx) % (2**32)
            seeds_used.append(current_seed)
            
            # Gradio progress callback
            def cb(step_idx, latent, is_start, override_num_inference_steps=None, **kwargs):
                pct = (step_idx + 1) / int(steps)
                total_pct = (idx + pct) / num_images
                progress(min(total_pct, 0.99), desc=f"Generating Image {idx+1}/{num_images} - Step {step_idx+1}/{steps}")
                
            print(f"Generating image {idx+1}/{num_images}: {width}x{height}, steps={steps}, seed={current_seed}")
            sys.stdout.flush()
            
            # Call Krea model generator
            result = krea_model.generate(
                seed=current_seed,
                input_prompt=prompt,
                n_prompt=negative_prompt if negative_prompt.strip() else None,
                sampling_steps=int(steps),
                width=width,
                height=height,
                guide_scale=0.0,
                batch_size=1,
                callback=cb,
                loras_slists={"phase1": []},
                rebalance_weights=weights_tensor,
            )
            
            if result is None:
                raise RuntimeError(f"Generation of image {idx+1} failed (returned None).")
                
            # Convert output tensor (shape: [3, 1, height, width] on CPU) to PIL Image
            img_np = result[:, 0].permute(1, 2, 0).numpy()  # shape: [height, width, 3]
            image = Image.fromarray(img_np)
            images.append(image)
            
            # Yield intermediate results to gallery with preview index forced to 0
            yield gr.update(value=images, selected_index=0), f"⏳ Generated {len(images)}/{num_images} images...", gr.update(visible=False)
            
        # Zip images if multiple images were generated
        zip_path = None
        if num_images > 1:
            import zipfile
            import tempfile
            temp_zip = tempfile.NamedTemporaryFile(suffix=".zip", delete=False)
            zip_path = temp_zip.name
            with zipfile.ZipFile(zip_path, 'w') as zipf:
                for idx, img in enumerate(images):
                    temp_img_path = tempfile.mktemp(suffix=".png")
                    img.save(temp_img_path)
                    zipf.write(temp_img_path, f"krea_image_{idx+1}_seed_{seeds_used[idx]}.png")
                    try:
                        os.remove(temp_img_path)
                    except Exception:
                        pass
                        
        elapsed = time.time() - start_time
        status = f"✅ Generated {num_images} image(s) in {elapsed:.2f}s | Seeds: {seeds_used} | Resolution: {width}x{height} | Accelerator: GPU T4 x2"
        yield gr.update(value=images, selected_index=0), status, gr.update(value=zip_path, visible=(zip_path is not None))
    except Exception as e:
        traceback.print_exc()
        yield gr.update(value=None, selected_index=None), f"❌ Error: {str(e)}", gr.update(visible=False)


# ==== GRADIO UI (AIQUEST BRANDED) ====
CSS = """@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1100px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(30,60,114,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.social-buttons { display: flex; justify-content: center; gap: 12px; flex-wrap: wrap; }
.social-btn { padding: 10px 24px; border-radius: 8px; font-weight: 700; font-size: 15px; text-decoration: none; display: inline-block; color: white; transition: all 0.3s; box-shadow: 0 4px 12px rgba(0,0,0,0.2); }
.social-btn:hover { transform: translateY(-2px); box-shadow: 0 6px 16px rgba(0,0,0,0.3); }
.youtube-btn { background: linear-gradient(135deg, #FF0000 0%, #CC0000 100%); }
.x-btn { background: linear-gradient(135deg, #000000 0%, #333333 100%); }
button.primary { background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
#clear-btn { background: linear-gradient(135deg, #6b7280 0%, #374151 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; }
"""

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="Krea-2-Turbo Image Generator | AIQUEST") as demo:
    gr.HTML('<div class="brand-header"><div class="brand-title">🎨 Krea-2-Turbo - Fast Text-to-Image Generator</div><div class="brand-subtitle">Created by <strong>AIQUEST Academy</strong> | Kaggle GPU T4 x2</div><div class="social-buttons"><a href="https://youtube.com/@aiquestacademy" target="_blank" class="social-btn youtube-btn">▶️ Subscribe on YouTube</a><a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a></div></div>')

    gr.Markdown(
        "Generate stunning native high-resolution images in seconds using the distilled **Krea-2-Turbo** model (Wan2GP INT8 Edition).\n\n"
        "**Settings optimized for speed:** Uses 8 steps with guidance scale disabled (CFG = 0.0) automatically."
    )

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Row(equal_height=True):
                prompt = gr.Textbox(
                    label="📝 Prompt", 
                    lines=3,
                    placeholder="A small, dark-colored cat is captured mid-stride, walking down the center of a narrow, abandoned street.",
                    scale=4
                )
                enhance_btn = gr.Button("✨ Enhance", variant="secondary", scale=1)
            
            with gr.Row():
                style_preset = gr.Dropdown(
                    label="🎨 Style Preset",
                    choices=["None", "Cinematic", "Photographic", "Anime", "Cyberpunk", "Fantasy"],
                    value="None"
                )
                resolution = gr.Dropdown(
                    label="🖥️ Base Resolution",
                    choices=["1024px (Standard)", "1536px (High)", "2048px (2K Ultra)"],
                    value="1024px (Standard)",
                    info="Note: 2K resolution may run slower or hit memory limits depending on T4 VRAM load"
                )
                
            aspect_ratio = gr.Dropdown(
                label="📏 Aspect Ratio",
                choices=["16:9 Landscape", "9:16 Portrait", "1:1 Square", "4:3 Standard", "3:4 Portrait"],
                value="16:9 Landscape"
            )
            
            with gr.Row():
                steps = gr.Slider(
                    label="⚡ Diffusion Steps",
                    minimum=1, maximum=12, step=1, value=8,
                    info="8 steps is recommended."
                )
                num_images = gr.Slider(
                    label="🖼️ Number of Images",
                    minimum=1, maximum=4, step=1, value=1,
                    info="Up to 4 images (sequential to avoid VRAM OOM)."
                )
            
            with gr.Accordion("⚙️ Advanced Settings", open=False):
                negative_prompt = gr.Textbox(
                    label="🚫 Negative Prompt",
                    placeholder="low quality, blurry, distorted, bad proportions, bad anatomy",
                    lines=2
                )
                with gr.Row():
                    seed = gr.Number(
                        label="🎲 Seed (-1 for Random)", 
                        value=-1, 
                        precision=0
                    )
                    bypass_safety = gr.Checkbox(
                        label="🔓 Bypass Safety Filter",
                        value=False
                    )
                with gr.Row():
                    bypass_multiplier = gr.Slider(
                        label="⚡ Guidance Multiplier",
                        minimum=1.0, maximum=10.0, step=0.1, value=4.0,
                        info="Global text conditioning amplification. 4.0 is default for safety bypass."
                    )
                    rebalance_weights = gr.Textbox(
                        label="⚖️ Rebalance Layer Weights",
                        value="1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.2,5.5,0.1,1.4,0.1",
                        lines=1,
                        info="Modify weights for Qwen's 12 selected layers to bypass restrictions."
                    )
            
            with gr.Row():
                gen_btn = gr.Button("🎨 Generate", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg", elem_id="clear-btn")

        with gr.Column(scale=1):
            image_out = gr.Gallery(
                label="🖼️ Generated Images", 
                columns=2, 
                rows=2, 
                object_fit="contain", 
                preview=True
            )
            zip_out = gr.File(label="📦 Download All Images (ZIP)", visible=False)
            status_out = gr.Textbox(label="ℹ️ Status", interactive=False)

    # Helper to enhance prompt manually
    def manual_enhance_trigger(p, s):
        return enhance_prompt_only_if_checked(p, True, s)

    # Wire up UI functions
    enhance_btn.click(
        fn=manual_enhance_trigger,
        inputs=[prompt, style_preset],
        outputs=[prompt]
    )

    gen_event = gen_btn.click(
        fn=generate_image,
        inputs=[prompt, negative_prompt, steps, aspect_ratio, resolution, seed, num_images, bypass_safety, bypass_multiplier, rebalance_weights],
        outputs=[image_out, status_out, zip_out]
    )
    
    clear_btn.click(
        fn=lambda: ("", "", 8, "16:9 Landscape", "1024px (Standard)", -1, 1, "None", False, 4.0, "1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.2,5.5,0.1,1.4,0.1", gr.update(value=None, selected_index=None), "", gr.update(visible=False)),
        inputs=[],
        outputs=[prompt, negative_prompt, steps, aspect_ratio, resolution, seed, num_images, style_preset, bypass_safety, bypass_multiplier, rebalance_weights, image_out, status_out, zip_out]
    )

    gr.HTML('<div class="footer"><p style="font-size: 16px; margin: 5px 0;">🎨 Created by <strong>AIQUEST Academy</strong></p><p style="font-size: 14px; margin: 5px 0; color: #9ca3af;">Free & Open Source | Krea-2-Turbo 12B | Kaggle GPU T4 x2</p><p style="font-size: 13px; margin: 10px 0;"><a href="https://youtube.com/@aiquestacademy" target="_blank" style="color: #1e3c72; text-decoration: none; margin: 0 10px;">YouTube</a> | <a href="https://x.com/aiquestacademy" target="_blank" style="color: #1e3c72; text-decoration: none; margin: 0 10px;">X (Twitter)</a> | <a href="https://aiquest.site" target="_blank" style="color: #1e3c72; text-decoration: none; margin: 0 10px;">aiquest.site</a></p></div>')

print("Launching Gradio interface...")
demo.queue()
demo.launch(share=True, inline=False, debug=True, show_error=True)

---
## Step 5: Launch! 🚀

In [ ]:
!python -u run_krea_turbo.py 2>&1